##### MCORR AND CNMF
**Analysis of riboL1-GCaMP8m responses imaged with 25x objective at zoom 1x over 765x765 pixels**

#### Define parameters


In [ ]:
# Paths
import json
from pathlib import Path
data_path = [Path('F:/Data/2P/C57_O1M1/09182023/TSeries-09182023-1503-003')]
export_path = Path('H:/Analysis/2P/C57_O1M1/09182023/run3/mesmerize_wo_zcorr')

# Motion correction parameters

params_mcorr = \
{
  'main':
    {
        'strides': [48, 48],
        'overlaps': [36, 36],
        'max_shifts': [18, 12],
        'max_deviation_rigid': 9,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# CNMF parameters
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 20, # framerate, very important!
            'p': 1,
            'nb': 1,
            'merge_thr': 0.8,
            'rf': 40,
            'stride': 20, # "stride" for cnmf, "strides" for mcorr
            'K': 12,
            'gSig': [8, 8],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.5,
            'SNR_lowest':  1.5,
            'rval_thr': 0.85,
            'rval_lowest': 0.2,
            'use_cnn': True,
            'min_cnn_thr': 0.95,
            'cnn_lowest': 0.2,
            'decay_time': 0.15,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

# Extra parameters
params_extra = \
    {
        'cleanup': False # If `True`, run cleanup after CNMF, i.e., delete all df data
    }

# Concatenate the two dictionaries
params = {'params_mcorr': params_mcorr, 'params_cnmf': params_cnmf, 'params_extra': params_extra}

# Add export_path to params dictionary
params['export_path'] = str(export_path)

# Add z correction parameters
zstack_path="F:/Data/2P/C57_O1M1/09182023/ZSeries-09182023-1503-003"
z_params_path="D:/Code/Analysis_2P/Mesmerize/parameters/params_zshift_C57_O1M1.json"

# Load z-corr parameters
with open(z_params_path) as f:
    zstack_params = json.load(f)
                    
# Add zstack_path, zstack_params to params dictionary
params['zstack_path'] = str(zstack_path)
params['z_params'] = zstack_params
# Override value in z_params, just in case: don't run pbp zcorr, just do z-correlation 
params['z_params']['do_zcorr'] = False

print(params)

#### Run MCORR and CNMF

In [ ]:
import pipeline_mcorr_cnmf as preproc
_ , batch_path = preproc.run_mcorr_cnmf(data_path, params)

### Load outputs

In [ ]:
# If cleanup was set to false in the params, you can load the batch file into Mesmerize:
# batch_path=Path('H:/Analysis/2P/C57_O1M2/10022023/run2/mesmerize_zcorr_fixSNR/batch_20240409-161715.pickle')
if params['params_extra']['cleanup'] == False:
    import mesmerize_core as mc
    df = mc.load_batch(batch_path)
df

### Visualize with fastplotlib


In [ ]:
import fastplotlib as fpl
cnmf_index = 1
rcm = df.iloc[cnmf_index].cnmf.get_rcm()
rcb = df.iloc[cnmf_index].cnmf.get_rcb()
residuals = df.iloc[cnmf_index].cnmf.get_residuals()
input_movie = df.iloc[cnmf_index].caiman.get_input_movie()

iw_rcm = fpl.ImageWidget(
    data=[input_movie, rcm, rcb, residuals], 
    grid_plot_kwargs={"size": (800, 600)}, 
    cmap="gnuplot2"
)
iw_rcm.show()

In [ ]:
iw_rcm.close()

### Visualize with `mesmerize-viz`

In [ ]:
print(f"Num accepted/rejected: {len(df.iloc[-1].cnmf.get_good_components())}, {len(df.iloc[-1].cnmf.get_bad_components())}")

In [ ]:
import mesmerize_viz
viz_simple = df.cnmf.viz(
    image_data_options=["max"],
    temporal_kwargs={"add_residuals": True}
)
viz_simple.show(sidecar=True)
# viz_simple.show()
viz_simple.image_widget.cmap = "gray"

In [ ]:
viz_simple?

In [ ]:
viz_simple.close()

#### Clean up export folder: stop/restart notebook, then run first and last cells

In [ ]:
import pipeline_mcorr_cnmf as preproc
batch_path=Path('H:/Analysis/2P/C57_O1M1/09182023/run2/mesmerize/batch_20240415-111637.pickle')
preproc.cleanup_files(batch_path, export_path)